## SafeDerm — 02. Data Understanding & Split

Inspects the metadata for structure, missing values, and duplicates, then produces the lesion-level train/val/test split every downstream notebook depends on.

Split is grouped by `lesion_id` — confirmed in `01_data_extraction.ipynb` that 1,956 lesions have more than one photo. No lesion's photos may span more than one split.

This notebook only understands and splits — it does not clean or impute. That's a later stage's job.

Run this once. Everything downstream reads from `data/processed/{train,val,test}_split.csv`.

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split

from src.config import (
    METADATA_PATH,
    PROCESSED_DATA_DIR,
    TRAIN_SPLIT_PATH,
    VAL_SPLIT_PATH,
    TEST_SPLIT_PATH,
)

### Load and inspect

Basic shape check, missing values, and exact duplicate rows — before we trust this data enough to split it.

In [3]:
df = pd.read_csv(METADATA_PATH)

print(f"Rows    : {len(df)}")
print(f"Columns : {list(df.columns)}")
print(f"\nMissing values:\n{df.isna().sum()}")
print(f"\nExact duplicate rows: {df.duplicated().sum()}")

Rows    : 10015
Columns : ['lesion_id', 'image_id', 'dx', 'dx_type', 'age', 'sex', 'localization']

Missing values:
lesion_id        0
image_id         0
dx               0
dx_type          0
age             57
sex              0
localization     0
dtype: int64

Exact duplicate rows: 0


`age` typically has a handful of missing values in this dataset — expected, not a bug. We flag it here and decide how to handle it (drop vs impute) in a later notebook. This one doesn't touch it.

### Build the lesion-level table

`lesion_id` is the split unit, not `image_id`. Collapse to one row per lesion before splitting — a lesion's `dx` (diagnosis) is the same across all its photos, so `.first()` is safe here.

In [4]:
lesion_df = df.groupby("lesion_id")["dx"].first().reset_index()

print(f"Unique lesions: {len(lesion_df)}")
print(lesion_df["dx"].value_counts())

Unique lesions: 7470
dx
nv       5403
bkl       727
mel       614
bcc       327
akiec     228
vasc       98
df         73
Name: count, dtype: int64


### Split — 70/15/15, stratified by class

Splitting happens on lesions, not images. Stratified by `dx` so rare classes (`df`: 115 images, `vasc`: 142) don't disappear from val/test by bad luck. `random_state=42` so this split is reproducible — every teammate who runs this gets the identical split.

In [5]:
train_lesions, temp_lesions = train_test_split(
    lesion_df,
    test_size=0.30,
    stratify=lesion_df["dx"],
    random_state=42,
)
val_lesions, test_lesions = train_test_split(
    temp_lesions,
    test_size=0.50,
    stratify=temp_lesions["dx"],
    random_state=42,
)

print(f"Train lesions: {len(train_lesions)}")
print(f"Val lesions  : {len(val_lesions)}")
print(f"Test lesions : {len(test_lesions)}")

Train lesions: 5229
Val lesions  : 1120
Test lesions : 1121


### Map back to image-level rows

The split decision was made on lesions. Now pull every image belonging to each lesion into its assigned split — this is where a multi-photo lesion's several images all land together.

In [6]:
train_df = df[df["lesion_id"].isin(train_lesions["lesion_id"])].copy()
val_df = df[df["lesion_id"].isin(val_lesions["lesion_id"])].copy()
test_df = df[df["lesion_id"].isin(test_lesions["lesion_id"])].copy()

print(f"Train images: {len(train_df)}")
print(f"Val images  : {len(val_df)}")
print(f"Test images : {len(test_df)}")

Train images: 6981
Val images  : 1532
Test images : 1502


### Verify — no lesion crosses a split boundary

This is the actual safety check this whole notebook exists for. If any `lesion_id` appears in more than one split, the split is broken and nothing downstream can be trusted.

In [7]:
train_ids = set(train_df["lesion_id"])
val_ids = set(val_df["lesion_id"])
test_ids = set(test_df["lesion_id"])

assert not (train_ids & val_ids), "Lesion overlap between train and val!"
assert not (train_ids & test_ids), "Lesion overlap between train and test!"
assert not (val_ids & test_ids), "Lesion overlap between val and test!"
assert len(train_df) + len(val_df) + len(test_df) == len(df), "Row count mismatch after split."

print("No lesion overlap across splits — confirmed.")
print("\nClass distribution per split (%):")
for name, split_df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"\n{name} (n={len(split_df)}):")
    print((split_df["dx"].value_counts(normalize=True) * 100).round(1))

No lesion overlap across splits — confirmed.

Class distribution per split (%):

train (n=6981):
dx
nv       67.1
mel      11.1
bkl      11.1
bcc       5.2
akiec     3.2
vasc      1.4
df        1.0
Name: proportion, dtype: float64

val (n=1532):
dx
nv       66.4
mel      11.3
bkl      10.4
bcc       5.4
akiec     3.5
df        1.6
vasc      1.4
Name: proportion, dtype: float64

test (n=1502):
dx
nv       66.8
bkl      11.1
mel      11.1
bcc       4.7
akiec     3.5
vasc      1.4
df        1.3
Name: proportion, dtype: float64


### Save

Writes the three split CSVs to `data/processed/`. Everything from `03_eda.ipynb` onward reads from these — never from the raw metadata directly.

In [8]:
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

train_df.to_csv(TRAIN_SPLIT_PATH, index=False)
val_df.to_csv(VAL_SPLIT_PATH, index=False)
test_df.to_csv(TEST_SPLIT_PATH, index=False)

print("Saved splits to data/processed/.")

Saved splits to data/processed/.
